In [ ]:
from pathlib import Path
import subprocess


def build_doc_id(pdf_path: Path) ->str:
    folder = pdf_path.parent.name
    stem = pdf_path.stem
    return f"{folder}M{stem}"

def build_output_dir(pdf_path: Path, base_dir: Path) -> Path:
    return base_dir / pdf_path.parent.name / pdf_path.stem

def convert_pdf_to_jpgs(pdf_path: Path, output_dir: Path, dpi: int = 300):
    output_dir.mkdir(parents=True, exist_ok=True)
    output_prefix = output_dir / "page"

    cmd = [
        "pdftoppm",
        "-jpeg",
        "-r",
        str(dpi),
        str(pdf_path),
        str(output_prefix),
    ]

    subprocess.run(cmd, check=True)

    image_paths = sorted(output_dir.glob("page-*.jpg"))
    return image_paths

def build_manifest_record(pdf_path: Path, image_paths: list[Path]) -> dict:
    return {
        "doc_id": build_doc_id(pdf_path),
        "pdf_path": str(pdf_path),
        "file_name": pdf_path.name,
        "folder": pdf_path.parent.name,
        "page_count": len(image_paths),
        "image_paths": [str(p) for p in image_paths],
    }

In [ ]:
pdf_path = Path("data/Sunrisetek-1/4114769644.pdf")
base_dir = Path("outputs/images")

output_dir = build_output_dir(pdf_path, base_dir)
image_paths = convert_pdf_to_jpgs(pdf_path, output_dir, dpi=300)
record = build_manifest_record(pdf_path, image_paths)

record



In [ ]:
data_dir = Path("data")
pdf_files = sorted(data_dir.rglob("*.pdf"))
records = []

for pdf_path in pdf_files:
    output_dir = build_output_dir(pdf_path, base_dir)
    image_paths = convert_pdf_to_jpgs(pdf_path, output_dir, dpi=300)
    record = build_manifest_record(pdf_path, image_paths)
    records.append(record)
